https://colab\.research\.google\.com/drive/1nIn8FRLoc32txCevJbDRmeRLFIBXBQ\-K?usp=sharing

### use link

In [3]:
!pip install --upgrade neuralforecast


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.2/426.2 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.1/231.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape


/root/venv/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/venv/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-09 17:01:19,780	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-09 17:01:20,110	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, the

In [7]:
df = pd.read_parquet('/work/sample_hotels-1.parquet')


In [9]:
df['ds'] = pd.to_datetime(df['ds'])

df = df.query("unique_id not in ['hotel_77', 'hotel_28']")


In [11]:
test_df = df.groupby('unique_id').tail(28).reset_index(drop=True)
train_df = df.groupby('unique_id').apply(lambda x: x.iloc[:-28]).reset_index(drop=True)


In [13]:
df = train_df[['unique_id', 'ds', 'y']]


In [15]:
models = [
    NBEATS(
        h=28,
        input_size=56,
    ),
    NHITS(
        h=28,
        input_size=56,
    )
]


Seed set to 1
Seed set to 1


In [17]:
nf = NeuralForecast(
    models=models,
    freq='D'
)


In [19]:
nf.fit(train_df[['unique_id', 'ds', 'y']])

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
2026-05-09 17:01:50.782166: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-09 17:01:50.782202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-09 17:01:50.783516: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-09 17:01:50.790843: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the approp

In [21]:
cv_df = nf.cross_validation(
    df=train_df[['unique_id', 'ds', 'y']],
    h=28,
    step_size=28,
    n_windows=5
)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
4.8 K     Non-trainable params
2.6 M     Total params
10.231    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode
                                                                             /root/venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Epoch 99: 100%|██████████| 1/1 [00:00<00:00,  4.61it/s, v_num=32, train_loss_step=0.0525, tr

In [1]:
eval_cv = evaluate(
    cv_df,
    metrics=[mae, rmse, mape],
    models=['NBEATS', 'NHITS']
)
display(eval_cv)


NameError: name 'evaluate' is not defined

### I dont think we need this code block below

In [ ]:
forecast_nf = nf.predict()

results_nf = forecast_nf.merge(
    test_df,
    on=['unique_id', 'ds']
)

eval_nf = evaluate(
    results_nf,
    metrics=[mae, rmse, mape]
)

display(eval_nf)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=94252149-99d5-4a3a-8951-54d7f526c43a' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>